# Task Specific Tuning

# Goal
- Load model (vocab extension, continual A or continual B).
- Evaluate its ability to classify Bashkir news by topic in zero-shot, few-shot (3 examples) and after full fine‑tuning modes (like in LLaMaTurk).
- Compare the quality of different model configurations

## Imports

In [1]:
!pip install -q wandb datasets transformers accelerate bitsandbytes peft scikit-learn peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [2]:
import os
import torch
import wandb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset, Dataset

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import PeftModel
from tqdm import tqdm

In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))

## Model

We will use 3 models:

- **baseline**: model after vocab extension
- **experiment_a**: model after vocab extension + continual training
- **experiment_b**: model after vocab extension + embedding align + continual training

We will evaluate them all using:
- zero-shot
- few-shot
- fine-tuning

In [5]:
#  baseline (vocab extension):
# MODEL_ARTIFACT = "llama2_bashkir_vocab"
# WANDB_PROJECT_SOURCE = "bashllama-vocab-extension"

# experiment A (continual NO alignment):
# MODEL_ARTIFACT = "llama2_bashkir_continual_A"
# WANDB_PROJECT_SOURCE = "bashllama-continual-training-A"

# experiment B (continual WITH alignment):
# MODEL_ARTIFACT = "llama2_bashkir_continual_B"
# WANDB_PROJECT_SOURCE = "bashllama-continual-training-B"

MODEL_ARTIFACT = "llama2_bashkir_continual_A"
WANDB_PROJECT_SOURCE = "bashllama-continual-training-A"

# log results
WANDB_LOG_PROJECT = "bashllama-task-specific"
WANDB_RUN_NAME = f"task_tuning_{MODEL_ARTIFACT}"

print(f"Using model: {MODEL_ARTIFACT} from project {WANDB_PROJECT_SOURCE}")

Using model: llama2_bashkir_continual_A from project bashllama-continual-training-A


In [6]:
temp_run = wandb.init(entity="e278979-metu-middle-east-technical-university",
                      project=WANDB_PROJECT_SOURCE,
                      name="temp_load_model")
artifact = temp_run.use_artifact(f"e278979-metu-middle-east-technical-university/{WANDB_PROJECT_SOURCE}/{MODEL_ARTIFACT}:latest",
                                 type="model")
model_dir = artifact.download()
temp_run.finish()

wandb: setting up run m1on7r5i
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260530_190846-m1on7r5i
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run temp_load_model
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-A
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-A/runs/m1on7r5i
wandb:   5 of 5 files downloaded.  
wandb: updating run metadata
wandb: uploading summary, console lines 0-0
wandb: 🚀 View run temp_load_model at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-A/runs/m1on7r5i
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-A
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260530_1

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

temp_run_base = wandb.init(entity="e278979-metu-middle-east-technical-university",
                           project="bashllama-vocab-extension",
                           name="temp_base_model")
base_artifact = temp_run_base.use_artifact("e278979-metu-middle-east-technical-university/bashllama-vocab-extension/llama2_bashkir_vocab:latest",
                                           type="model")
base_model_dir = base_artifact.download()
temp_run_base.finish()

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    use_cache=False
)
model = PeftModel.from_pretrained(base_model, model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model {MODEL_ARTIFACT} downloaded")

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260530_190852-ao7zurrw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run temp_base_model
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/ao7zurrw
wandb: Downloading large artifact 'llama2_bashkir_vocab:latest', 4142.38MB. 6 files...
wandb:   6 of 6 files downloaded.  
Done. 00:00:40.6 (102.1MB/s)
wandb: updating run metadata
wandb: uploading summary, console lines 1-2
wandb: 🚀 View run temp_base_model at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/ao7zurrw
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model llama2_bashkir_continual_A downloaded


In [8]:
wandb.init(project=WANDB_LOG_PROJECT, name=WANDB_RUN_NAME)

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260530_190949-8zhkfe9w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run task_tuning_llama2_bashkir_continual_A
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-task-specific
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-task-specific/runs/8zhkfe9w


## Dataset

In [9]:
dataset = load_dataset("metuKKhud/bashqort-task", split="train")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/267 [00:00<?, ? examples/s]

In [10]:
texts = dataset['title']
labels = dataset['topic']

label_list = sorted(set(labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

Divide to train and test

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"train: {len(X_train)} rows")
print(f"test: {len(X_test)} rows")
print(f"labels: {label_list}")

train: 213 rows
test: 54 rows
labels: ['Иҡтисад', 'Мәғариф', 'Мәҙәниәт', 'Социаль өлкә', 'Спорт', 'Сәйәсәт', 'Хәрби хеҙмәт', 'Хәүефһеҙлек', 'Ғәҙәттән тыш хәлдәр', 'Һаулыҡ һаҡлау']


## N-Shot evaluation

### Generate promts

In [12]:
def format_prompt(text, examples=None):
    instruction = ("Яңылыҡтың темаһын билдәлә. Мөмкин булған темалар: " + 
                   ", ".join(label_list) + 
                   "\nЯуап тик бер һүҙҙән торорға тейеш (бер генә теманы яҙығыҙ).")
    prompt = f"### Күрһәтмә:\n{instruction}\n\n"
    if examples:
        for ex_text, ex_label in examples:
            prompt += f"### Миҫал:\nТекст: {ex_text}\nТема: {ex_label}\n\n"
    prompt += f"### Текст:\n{text}\n### Тема:"
    return prompt

def predict_topic(model, tokenizer, text, examples=None, max_new_tokens=20):
    prompt = format_prompt(text, examples=examples)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # search 1st "### Тема:"
    if "### Тема:" in generated:
        parts = generated.split("### Тема:", 1)
        if len(parts) > 1:
            answer = parts[1].strip().split("\n")[0].split()[0]
        else:
            answer = ""
    else:
        answer = ""
    for label in label_list:
        if answer.lower().startswith(label.lower()) or label.lower() in answer.lower():
            return label
    return answer


### Evaluate (0-shot)!

In [13]:
model.eval()
zero_preds = []
print("Zero-shot predictions (first 10):")
for i, text in enumerate(tqdm(X_test, desc="Zero-shot")):
    pred = predict_topic(model, tokenizer, text)
    zero_preds.append(pred)
    if i < 10:
        print(f"{i}: pred = {pred:15} | true = {y_test[i]}")
zero_acc = accuracy_score(y_test, zero_preds)
print(f"Zero-shot accuracy: {zero_acc:.4f}")

Zero-shot predictions (first 10):


Zero-shot:   2%|▏         | 1/54 [00:04<03:56,  4.46s/it]

0: pred = Ҡыҙыл           | true = Мәғариф


Zero-shot:   4%|▎         | 2/54 [00:07<03:22,  3.90s/it]

1: pred = Ҡыҙыл           | true = Хәүефһеҙлек


...

8: pred = Икенсе          | true = Социаль өлкә


Zero-shot:  19%|█▊        | 10/54 [00:35<02:36,  3.56s/it]

9: pred = Ҡатын-ҡыҙҙарҙың | true = Спорт


Zero-shot: 100%|██████████| 54/54 [03:19<00:00,  3.70s/it]

Zero-shot accuracy: 0.0000


In [14]:
wandb.log({"zero_shot_accuracy": zero_acc})

### Evaluate(3-shot)!

In [15]:
few_examples = list(zip(X_train[:3], y_train[:3]))
few_preds = []
print("Few-shot (3) predictions (first 10):")
for i, text in enumerate(tqdm(X_test, desc="Few-shot (3)")):
    pred = predict_topic(model, tokenizer, text, examples=few_examples)
    few_preds.append(pred)
    if i < 10:
        print(f"{i}: pred = {pred:15} | true = {y_test[i]}")
few_acc = accuracy_score(y_test, few_preds)
print(f"Few-shot (3) accuracy: {few_acc:.4f}")

Few-shot (3) predictions (first 10):


Few-shot (3):   2%|▏         | 1/54 [00:05<04:44,  5.36s/it]

0: pred = Мәғариф         | true = Мәғариф


Few-shot (3):   4%|▎         | 2/54 [00:10<04:38,  5.35s/it]

1: pred = Хәүефһеҙлек     | true = Хәүефһеҙлек


...

8: pred = Хәүефһеҙлек     | true = Социаль өлкә


Few-shot (3):  19%|█▊        | 10/54 [00:52<03:49,  5.21s/it]

9: pred = Хәүефһеҙлек     | true = Спорт


Few-shot (3): 100%|██████████| 54/54 [04:43<00:00,  5.26s/it]

Few-shot (3) accuracy: 0.1111


In [16]:
wandb.log({"3_shot_accuracy": few_acc})

## Fine Tuning

3 epochs, early stopping by evalloss

### Generate prompts

In [17]:
def create_finetune_example(text, label):
    prompt = format_prompt(text)
    return prompt + f" {label}\n"

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

In [18]:
train_texts = [create_finetune_example(t, l) for t, l in zip(X_train, y_train)]
test_texts  = [create_finetune_example(t, l) for t, l in zip(X_test, y_test)]

train_dataset = Dataset.from_dict({"text": train_texts})
test_dataset  = Dataset.from_dict({"text": test_texts})

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_tokenized  = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

train_tokenized.set_format("torch", columns=["input_ids", "attention_mask"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask"])

print(f"Training dataset size: {len(train_tokenized)}")
print(f"Test dataset size: {len(test_tokenized)}")

Map:   0%|          | 0/213 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Training dataset size: 213
Test dataset size: 54


### Train!

In [19]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,975,090,688 || trainable%: 0.0601


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:
training_args = TrainingArguments(
    output_dir="./task_checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=2,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    report_to=["wandb"],
    run_name=WANDB_RUN_NAME,
    remove_unused_columns=False,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss,Validation Loss


TrainOutput(global_step=42, training_loss=1.0220186284610204, metrics={'train_runtime': 634.7655, 'train_samples_per_second': 1.007, 'train_steps_per_second': 0.066, 'total_flos': 6603341316489216.0, 'train_loss': 1.0220186284610204, 'epoch': 3.0})

### Evaluate

In [22]:
model.eval()
finetune_preds = []
print("Fine-tuned predictions (first 10):")
for i, text in enumerate(tqdm(X_test, desc="Fine-tuned")):
    pred = predict_topic(model, tokenizer, text)
    finetune_preds.append(pred)
    if i < 10:
        print(f"{i}: pred = {pred:15} | true = {y_test[i]}")
finetune_acc = accuracy_score(y_test, finetune_preds)
print(f"Fine-tuned accuracy: {finetune_acc:.4f}")

Fine-tuned predictions (first 10):


Fine-tuned:   2%|▏         | 1/54 [00:03<03:14,  3.66s/it]

0: pred = Мәҙәниәт        | true = Мәғариф


Fine-tuned:   4%|▎         | 2/54 [00:07<03:11,  3.68s/it]

1: pred = Һаулыҡ          | true = Хәүефһеҙлек


...

8: pred = Мәҙәниәт        | true = Социаль өлкә


Fine-tuned:  19%|█▊        | 10/54 [00:37<02:47,  3.80s/it]

9: pred = Спорт           | true = Спорт


Fine-tuned: 100%|██████████| 54/54 [03:22<00:00,  3.74s/it]

Fine-tuned accuracy: 0.2778


In [23]:
wandb.log({
    "zero_shot_accuracy": zero_acc,
    "few_shot_3_accuracy": few_acc,
    "fine_tuned_accuracy": finetune_acc,
    "model_artifact": MODEL_ARTIFACT
})

output_dir = f"/kaggle/working/task_tuned_{MODEL_ARTIFACT}"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

artifact = wandb.Artifact(
    name=f"task_tuned_{MODEL_ARTIFACT}",
    type="model",
    description=f"Task-specific tuning on bashqort-task for {MODEL_ARTIFACT}"
)
artifact.add_dir(output_dir)
wandb.log_artifact(artifact)

print(f"Results:\nZero-shot: {zero_acc:.4f}\nFew-shot: {few_acc:.4f}\nFine-tuned: {finetune_acc:.4f}")

wandb.finish()

wandb: Adding directory to artifact (/kaggle/working/task_tuned_llama2_bashkir_continual_A)... Done. 0.1s
wandb: uploading artifact task_tuned_llama2_bashkir_continual_A; updating run metadata


Results:
Zero-shot: 0.0000
Few-shot: 0.1111
Fine-tuned: 0.2778


wandb: uploading artifact task_tuned_llama2_bashkir_continual_A
wandb: uploading history steps 24-24, summary, console lines 42-42
wandb: 
wandb: Run history:
wandb:     3_shot_accuracy ▁
wandb: few_shot_3_accuracy ▁
wandb: fine_tuned_accuracy ▁
wandb:         train/epoch ▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
wandb:   train/global_step ▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇████
wandb:     train/grad_norm ▂▂▃▃▃▄█▃▂▂▁▂▁▁▁▁▁▁▁▁▂
wandb: train/learning_rate ██▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▁▁
wandb:          train/loss █▇▆▅▄▃▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁
wandb:  zero_shot_accuracy ▁▁
wandb: 
wandb: Run summary:
wandb:     3_shot_accuracy 0.11111
wandb: few_shot_3_accuracy 0.11111
wandb: fine_tuned_accuracy 0.27778
wandb:      model_artifact llama2_bashkir_conti...
wandb:          total_flos 6603341316489216.0
wandb:         train/epoch 3
wandb:   train/global_step 42
wandb:     train/grad_norm 0.90644
wandb: train/learning_rate 0.0
wandb:          train/loss 0.5111
wandb:                  +5 ...
wandb: 
wandb: 🚀 View run task_tuning_llama2_bashkir_con

In [24]:
for i in range(5):
    print(f"Input: {X_test[i]}")
    prompt = format_prompt(X_test[i])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.0, do_sample=False)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Full output:\n{generated}")
    print("---")

Input: Башҡортостанда «Ауыл педагогы» программаһын тормошҡа ашырыу күҙаллана
Full output:
### Күрһәтмә:
Яңылыҡтың темаһын билдәлә. Мөмкин булған темалар: Иҡтисад, Мәғариф, Мәҙәниәт, Социаль өлкә, Спорт, Сәйәсәт, Хәрби хеҙмәт, Хәүефһеҙлек, Ғәҙәттән тыш хәлдәр, Һаулыҡ һаҡлау
Яуап тик бер һүҙҙән торорға тейеш(бер генә теманы яҙығыҙ).

### Текст:
Башҡортостанда«Ауыл педагогы» программаһын тормошҡа ашырыу күҙаллана
### Тема: Мәҙәниәт
### Хәүефһеҙлек: Мәҙән Ғаҙәт Хәбиров
Яуап темаһ
---
Input: Республикала балалар баҡсаларында һәм мәктәптәрҙә хәүефһеҙлек тикшерелә
Full output:
### Күрһәтмә:
Яңылыҡтың темаһын билдәлә. Мөмкин булған темалар: Иҡтисад, Мәғариф, Мәҙәниәт, Социаль өлкә, Спорт, Сәйәсәт, Хәрби хеҙмәт, Хәүефһеҙлек, Ғәҙәттән тыш хәлдәр, Һаулыҡ һаҡлау
Яуап тик бер һүҙҙән торорға тейеш(бер генә теманы яҙығыҙ).

### Текст:
Республикала балалар баҡсаларында һәм мәктәптәрҙә хәүефһеҙлек тикшерелә
### Тема: Һаулыҡ һаҡлау
### Тема: Хәүефһеҙлек
### Тема: Мәҙәниәт
### Тема
---
Input: Ата-әсәләр 